# Day 2: SQL analysis

**Business question:** which customers default, how do they get there, and what early-warning signals should a digital lender act on?

This notebook runs every Day 2 query against `credit.db`. Each query is shown in full so it reads like a report. The first time you run a query, it is also saved to `sql/` as a `.sql` file (existing files are never overwritten, so your edits and insights are safe).

Run the cells from top to bottom. After each result, fill in the **Insight** line in the markdown cell and in the matching `.sql` file.

In [20]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "processed" / "credit.db"
SQL_DIR = ROOT / "sql"
SQL_DIR.mkdir(exist_ok=True)

assert DB_PATH.exists(), f"Database not found at {DB_PATH}. Rerun the Day 1 loading step."
conn = sqlite3.connect(DB_PATH)

results = {}   # filename -> result table, used for the CSV export at the end


def run_query(filename, query):
    """Run a query and show its result. Saves it to sql/ if the file doesn't exist yet."""
    query = query.strip() + "\n"
    path = SQL_DIR / filename
    if not path.exists():
        path.write_text(query)
        print(f"[saved] sql/{filename}")
    out = pd.read_sql(query, conn)
    results[filename] = out
    return out

## Step 0: Check the raw table
Confirm the column names match what the queries expect. In the UCI file, `pay_0` is September (the most recent month) and there is no `pay_1`.

In [21]:
cols = pd.read_sql("PRAGMA table_info(clients)", conn)["name"].tolist()

needed = (["id", "limit_bal", "sex", "education", "marriage", "age", "pay_0"]
          + [f"pay_{i}" for i in range(2, 7)]
          + [f"bill_amt{i}" for i in range(1, 7)]
          + [f"pay_amt{i}" for i in range(1, 7)]
          + ["default_payment_next_month"])
missing = [c for c in needed if c not in cols]
assert not missing, f"These columns are missing from the clients table: {missing}. Compare with df.columns from Day 1."

pd.read_sql("SELECT COUNT(*) AS rows_in_raw_table FROM clients", conn)

,rows_in_raw_table
0,30000


## 00: Clean view
Readable column names, labelled categories (education 0/5/6 and marriage 0 become "Others/Unknown"), and age bands. A view keeps the raw table untouched.

In [22]:
CLEAN_VIEW = '''
-- Question: how do we get one clean, readable version of the data to analyze?
DROP VIEW IF EXISTS clients_clean;
CREATE VIEW clients_clean AS
SELECT
  id,
  limit_bal,
  CASE sex WHEN 1 THEN 'Male' WHEN 2 THEN 'Female' END AS sex,
  CASE WHEN education = 1 THEN 'Graduate school'
       WHEN education = 2 THEN 'University'
       WHEN education = 3 THEN 'High school'
       ELSE 'Others/Unknown' END AS education,
  CASE WHEN marriage = 1 THEN 'Married'
       WHEN marriage = 2 THEN 'Single'
       ELSE 'Others/Unknown' END AS marital_status,
  age,
  CASE WHEN age < 25 THEN '<25'
       WHEN age < 35 THEN '25-34'
       WHEN age < 45 THEN '35-44'
       WHEN age < 55 THEN '45-54'
       ELSE '55+' END AS age_band,
  pay_0 AS pay_sep, pay_2 AS pay_aug, pay_3 AS pay_jul,
  pay_4 AS pay_jun, pay_5 AS pay_may, pay_6 AS pay_apr,
  bill_amt1 AS bill_sep, bill_amt2 AS bill_aug, bill_amt3 AS bill_jul,
  bill_amt4 AS bill_jun, bill_amt5 AS bill_may, bill_amt6 AS bill_apr,
  pay_amt1 AS paid_sep, pay_amt2 AS paid_aug, pay_amt3 AS paid_jul,
  pay_amt4 AS paid_jun, pay_amt5 AS paid_may, pay_amt6 AS paid_apr,
  default_payment_next_month AS is_default
FROM clients;
'''

view_path = SQL_DIR / "00_clean_view.sql"
if not view_path.exists():
    view_path.write_text(CLEAN_VIEW.strip() + "\n")
    print("[saved] sql/00_clean_view.sql")

conn.executescript(CLEAN_VIEW)
pd.read_sql("SELECT COUNT(*) AS rows_in_view FROM clients_clean", conn)

,rows_in_view
0,30000


In [23]:
print(view_path)

c:\Users\user\Desktop\Python_Data_Project\risk_credit_card_analysis\sql\00_clean_view.sql


The view should return the same number of rows as the raw table (30,000 for the UCI data).

## 01: Portfolio overview
**Question:** what is the overall default rate and average credit limit?

This is your baseline. Every segment below is compared against this default rate.

In [24]:
q01 = run_query("01_portfolio_overview.sql", '''
-- Question: what is the overall default rate and average credit limit?
-- Insight: (fill in after running)
SELECT COUNT(*) AS customers,
       SUM(is_default) AS defaults,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct,
       ROUND(AVG(limit_bal)) AS avg_limit
FROM clients_clean;
''')
q01

,customers,defaults,default_rate_pct,avg_limit
0,30000,6636,22.12,167484.0


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 02: Default rate by segment
**Question:** which segments carry the most default risk, and how much of all defaults does each one contribute?

Compare each segment's default rate with its share of total defaults. A small segment with a high rate matters less than a large segment with a moderately high rate.

In [25]:
q02 = run_query("02_default_by_segment.sql", '''
-- Question: which segments carry the most default risk, and how much of total defaults does each contribute?
-- Insight: (fill in after running)
SELECT 'education' AS dimension, education AS segment, COUNT(*) AS customers,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct,
       ROUND(100.0 * SUM(is_default) / (SELECT SUM(is_default) FROM clients_clean), 2) AS share_of_defaults_pct
FROM clients_clean GROUP BY education
UNION ALL
SELECT 'marital_status', marital_status, COUNT(*),
       ROUND(100.0 * AVG(is_default), 2),
       ROUND(100.0 * SUM(is_default) / (SELECT SUM(is_default) FROM clients_clean), 2)
FROM clients_clean GROUP BY marital_status
UNION ALL
SELECT 'age_band', age_band, COUNT(*),
       ROUND(100.0 * AVG(is_default), 2),
       ROUND(100.0 * SUM(is_default) / (SELECT SUM(is_default) FROM clients_clean), 2)
FROM clients_clean GROUP BY age_band
UNION ALL
SELECT 'sex', sex, COUNT(*),
       ROUND(100.0 * AVG(is_default), 2),
       ROUND(100.0 * SUM(is_default) / (SELECT SUM(is_default) FROM clients_clean), 2)
FROM clients_clean GROUP BY sex
ORDER BY dimension, default_rate_pct DESC;
''')
q02

,dimension,segment,customers,default_rate_pct,share_of_defaults_pct
0,age_band,<25,2685,27.19,11.00
1,age_band,55+,1053,26.69,4.23
2,age_band,45-54,4233,23.93,15.27
3,age_band,35-44,9018,21.86,29.70
4,age_band,25-34,13011,20.30,39.80
5,education,High school,4917,25.16,18.64
6,education,University,14030,23.73,50.18
7,education,Graduate school,10585,19.23,30.68
8,education,Others/Unknown,468,7.05,0.50
9,marital_status,Others/Unknown,377,23.61,1.34


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 03: Credit limit deciles (`NTILE`)
**Question:** do customers with higher or lower credit limits default more?

Many customers share the same limit, so decile boundaries can overlap slightly. The `id` in the `ORDER BY` makes the split repeatable. Mention the overlap in your insight.

In [26]:
q03 = run_query("03_limit_deciles.sql", '''
-- Question: do customers with higher or lower credit limits default more?
-- Insight: (fill in after running)
WITH ranked AS (
  SELECT *, NTILE(10) OVER (ORDER BY limit_bal, id) AS limit_decile
  FROM clients_clean
)
SELECT limit_decile,
       MIN(limit_bal) AS min_limit, MAX(limit_bal) AS max_limit,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct
FROM ranked
GROUP BY limit_decile
ORDER BY limit_decile;
''')
q03

,limit_decile,min_limit,max_limit,customers,default_rate_pct
0,1,10000,30000,3000,35.97
1,2,30000,50000,3000,29.93
2,3,50000,70000,3000,28.07
3,4,70000,100000,3000,24.57
4,5,100000,140000,3000,23.10
5,6,140000,180000,3000,18.30
6,7,180000,210000,3000,17.40
7,8,210000,270000,3000,16.40
8,9,270000,360000,3000,14.87
9,10,360000,1000000,3000,12.60


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 04: Utilization bands
**Question:** how does default risk change as customers use more of their credit limit?

Utilization is the September bill divided by the credit limit. Some bills are negative (credit balances), which is why the first band is `<= 0`.

In [27]:
q04 = run_query("04_utilization_bands.sql", '''
-- Question: how does default risk change as customers use more of their credit limit?
-- Insight: (fill in after running)
WITH util AS (
  SELECT *, 1.0 * bill_sep / limit_bal AS utilization FROM clients_clean
)
SELECT CASE WHEN utilization <= 0 THEN '0. No balance'
            WHEN utilization < 0.3 THEN '1. Under 30%'
            WHEN utilization < 0.6 THEN '2. 30-59%'
            WHEN utilization < 0.9 THEN '3. 60-89%'
            ELSE '4. 90%+' END AS utilization_band,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct
FROM util
GROUP BY utilization_band
ORDER BY utilization_band;
''')
q04

,utilization_band,customers,default_rate_pct
0,0. No balance,2598,24.75
1,1. Under 30%,12194,16.93
2,2. 30-59%,4307,23.19
3,3. 60-89%,4851,27.58
4,4. 90%+,6050,26.30


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 05: Delinquency history
**Question:** how does the number of late months in the last six relate to default?

`(pay_sep >= 1) + (pay_aug >= 1) + ...` adds up true/false checks, so the result is the count of months a customer was at least one month late.

In [28]:
q05 = run_query("05_delinquency_history.sql", '''
-- Question: how does the number of late months in the last six relate to default?
-- Insight: (fill in after running)
WITH d AS (
  SELECT is_default,
         (pay_sep >= 1) + (pay_aug >= 1) + (pay_jul >= 1)
       + (pay_jun >= 1) + (pay_may >= 1) + (pay_apr >= 1) AS months_late
  FROM clients_clean
)
SELECT months_late,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct
FROM d
GROUP BY months_late
ORDER BY months_late;
''')
q05

,months_late,customers,default_rate_pct
0,0,19931,11.71
1,1,4426,29.82
2,2,1899,38.76
3,3,1154,50.87
4,4,951,57.31
5,5,298,57.38
6,6,1341,70.32


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

### 05b: Latest repayment status alone
This is usually the strongest single signal. Codes: -2 no consumption, -1 paid in full, 0 revolving (minimum paid), 1+ months late.

In [29]:
q05b = run_query("05b_latest_status.sql", '''
-- Question: how does the most recent repayment status relate to default?
-- Insight: (fill in after running)
SELECT pay_sep AS latest_status,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct
FROM clients_clean
GROUP BY pay_sep
ORDER BY pay_sep;
''')
q05b

,latest_status,customers,default_rate_pct
0,-2,2759,13.23
1,-1,5686,16.78
2,0,14737,12.81
3,1,3688,33.95
4,2,2667,69.14
5,3,322,75.78
6,4,76,68.42
7,5,26,50.00
8,6,11,54.55
9,7,9,77.78


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 06: Roll-rate matrix
**Question:** from one month to the next, what share of customers cure, stay late, or get worse?

This is the main query of the project. It compares each customer's August status (`from_state`) with September (`to_state`). `SUM(COUNT(*)) OVER (PARTITION BY from_state)` gives each starting state's total, so every row shows a percentage of its own starting group.

In [30]:
q06 = run_query("06_roll_rates.sql", '''
-- Question: from one month to the next, what share of customers cure, stay late, or get worse?
-- Insight: (fill in after running)
WITH states AS (
  SELECT
    CASE WHEN pay_aug <= 0 THEN '0_Current'
         WHEN pay_aug = 1 THEN '1_One month late'
         WHEN pay_aug = 2 THEN '2_Two months late'
         ELSE '3_Three+ months late' END AS from_state,
    CASE WHEN pay_sep <= 0 THEN '0_Current'
         WHEN pay_sep = 1 THEN '1_One month late'
         WHEN pay_sep = 2 THEN '2_Two months late'
         ELSE '3_Three+ months late' END AS to_state
  FROM clients_clean
)
SELECT from_state, to_state,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY from_state), 2) AS pct_of_from_state
FROM states
GROUP BY from_state, to_state
ORDER BY from_state, to_state;
''')
q06

,from_state,to_state,customers,pct_of_from_state
0,0_Current,0_Current,22735,88.94
1,0_Current,1_One month late,1836,7.18
2,0_Current,2_Two months late,991,3.88
3,1_One month late,1_One month late,28,100.00
4,2_Two months late,0_Current,392,9.98
5,2_Two months late,1_One month late,1672,42.58
6,2_Two months late,2_Two months late,1591,40.51
7,2_Two months late,3_Three+ months late,272,6.93
8,3_Three+ months late,0_Current,55,11.39
9,3_Three+ months late,1_One month late,152,31.47


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

### 06b: Roll rates across all month pairs (stretch)
One month pair can be noisy. This stacks all five consecutive pairs (Apr to May, May to Jun, ... Aug to Sep) with `UNION ALL` and measures them together, which gives a steadier matrix.

In [31]:
q06b = run_query("06b_roll_rates_all_months.sql", '''
-- Question: what are the average month-to-month roll rates across all five month pairs?
-- Insight: (fill in after running)
WITH pairs AS (
  SELECT pay_aug AS prev_status, pay_sep AS next_status FROM clients_clean
  UNION ALL SELECT pay_jul, pay_aug FROM clients_clean
  UNION ALL SELECT pay_jun, pay_jul FROM clients_clean
  UNION ALL SELECT pay_may, pay_jun FROM clients_clean
  UNION ALL SELECT pay_apr, pay_may FROM clients_clean
),
states AS (
  SELECT
    CASE WHEN prev_status <= 0 THEN '0_Current'
         WHEN prev_status = 1 THEN '1_One month late'
         WHEN prev_status = 2 THEN '2_Two months late'
         ELSE '3_Three+ months late' END AS from_state,
    CASE WHEN next_status <= 0 THEN '0_Current'
         WHEN next_status = 1 THEN '1_One month late'
         WHEN next_status = 2 THEN '2_Two months late'
         ELSE '3_Three+ months late' END AS to_state
  FROM pairs
)
SELECT from_state, to_state,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY from_state), 2) AS pct_of_from_state
FROM states
GROUP BY from_state, to_state
ORDER BY from_state, to_state;
''')
q06b

,from_state,to_state,customers,pct_of_from_state
0,0_Current,0_Current,123723,93.88
1,0_Current,1_One month late,1860,1.41
2,0_Current,2_Two months late,6209,4.71
3,1_One month late,1_One month late,34,100.00
4,2_Two months late,0_Current,4130,25.34
5,2_Two months late,1_One month late,1676,10.28
6,2_Two months late,2_Two months late,9460,58.05
7,2_Two months late,3_Three+ months late,1031,6.33
8,3_Three+ months late,0_Current,200,10.66
9,3_Three+ months late,1_One month late,152,8.10


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 07: Payment ratio
**Question:** do customers who pay only the minimum default more than those who pay most of the bill?

September's payment goes against the **August** statement, so the ratio is `paid_sep / bill_aug`. Customers with no balance due get a `NULL` ratio and their own band.

In [32]:
q07 = run_query("07_payment_ratio.sql", '''
-- Question: do customers who pay only the minimum default more than those who pay most of the bill?
-- Insight: (fill in after running)
WITH r AS (
  SELECT is_default,
         CASE WHEN bill_aug <= 0 THEN NULL ELSE 1.0 * paid_sep / bill_aug END AS pay_ratio
  FROM clients_clean
)
SELECT CASE WHEN pay_ratio IS NULL THEN '0. No balance due'
            WHEN pay_ratio < 0.05 THEN '1. Under 5% (minimum or less)'
            WHEN pay_ratio < 0.25 THEN '2. 5-24%'
            WHEN pay_ratio < 1 THEN '3. 25-99%'
            ELSE '4. Paid in full' END AS payment_band,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct
FROM r
GROUP BY payment_band
ORDER BY payment_band;
''')
q07

,payment_band,customers,default_rate_pct
0,0. No balance due,3175,23.43
1,1. Under 5% (minimum or less),9664,27.90
2,2. 5-24%,8452,22.75
3,3. 25-99%,1886,17.23
4,4. Paid in full,6823,13.89


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 08: Rule-based risk tiers
**Question:** if we split customers into High, Medium and Low risk using simple rules, how well do the tiers separate defaulters?

- **High:** two or more months late in September
- **Medium:** one month late, or utilization of 90% or more
- **Low:** everyone else

**Lift** is a tier's default rate divided by the portfolio rate. This table feeds your Day 4 intervention model. Adjust the thresholds after reviewing queries 04 to 07 if you see a better cut.

In [33]:
q08 = run_query("08_risk_tiers.sql", '''
-- Question: how well do simple High/Medium/Low rules separate defaulters?
-- Insight: (fill in after running)
WITH tiered AS (
  SELECT is_default,
         CASE WHEN pay_sep >= 2 THEN 'High'
              WHEN pay_sep = 1 OR 1.0 * bill_sep / limit_bal >= 0.9 THEN 'Medium'
              ELSE 'Low' END AS risk_tier
  FROM clients_clean
)
SELECT risk_tier,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM tiered), 2) AS pct_of_customers,
       ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct,
       ROUND(100.0 * SUM(is_default) / (SELECT SUM(is_default) FROM tiered), 2) AS pct_of_defaults,
       ROUND(AVG(is_default) / (SELECT AVG(is_default) FROM tiered), 2) AS lift
FROM tiered
GROUP BY risk_tier
ORDER BY default_rate_pct DESC;
''')
q08

,risk_tier,customers,pct_of_customers,default_rate_pct,pct_of_defaults,lift
0,High,3130,10.43,69.55,32.81,3.14
1,Medium,8151,27.17,24.29,29.84,1.10
2,Low,18719,62.40,13.24,37.36,0.60


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## 09: Rank segment combinations
**Question:** which age and education combinations have the highest default rates?

`HAVING COUNT(*) >= 200` removes small segments, because tiny groups produce noisy rates. Say so in your README.

In [34]:
q09 = run_query("09_segment_ranking.sql", '''
-- Question: which age and education combinations have the highest default rates?
-- Insight: (fill in after running)
WITH seg AS (
  SELECT age_band, education,
         COUNT(*) AS customers,
         SUM(is_default) AS defaults,
         ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct
  FROM clients_clean
  GROUP BY age_band, education
  HAVING COUNT(*) >= 200
)
SELECT *, RANK() OVER (ORDER BY default_rate_pct DESC) AS risk_rank
FROM seg
ORDER BY risk_rank;
''')
q09

,age_band,education,customers,defaults,default_rate_pct,risk_rank
0,55+,University,344,104,30.23,1
1,<25,High school,315,94,29.84,2
2,<25,University,1610,450,27.95,3
3,55+,Graduate school,258,68,26.36,4
4,45-54,High school,1323,341,25.77,5
5,<25,Graduate school,720,183,25.42,6
6,45-54,University,1791,451,25.18,7
7,35-44,High school,1581,390,24.67,8
8,55+,High school,440,108,24.55,9
9,25-34,High school,1258,304,24.17,10


> **Insight:** *(write 1-2 sentences with specific numbers after you run this)*

## Export results for Excel (Day 4)
Saves each result table to `data/processed/` as a CSV.

In [35]:
out = ROOT / "data" / "processed"
out.mkdir(parents=True, exist_ok=True)

for filename, table in results.items():
    csv_name = "q" + filename.replace(".sql", ".csv")
    table.to_csv(out / csv_name, index=False)

print(f"Saved {len(results)} CSV files to {out}")
sorted(p.name for p in out.glob("q*.csv"))

Saved 11 CSV files to c:\Users\user\Desktop\Python_Data_Project\risk_credit_card_analysis\data\processed


['q01_portfolio_overview.csv',
 'q02_default_by_segment.csv',
 'q02_segments.csv',
 'q03_limit_deciles.csv',
 'q04_utilization.csv',
 'q04_utilization_bands.csv',
 'q05_delinquency.csv',
 'q05_delinquency_history.csv',
 'q05b_latest_status.csv',
 'q06_roll_matrix.csv',
 'q06_roll_rates.csv',
 'q06b_roll_rates_all_months.csv',
 'q07_payment_ratio.csv',
 'q08_risk_tiers.csv',
 'q09_segment_ranking.csv']

## Day 2 checklist
- [ ] All queries run without errors
- [ ] Each Insight line is filled in, here and in the matching `.sql` file, with specific numbers
- [ ] The view returned the same number of rows as the raw table
- [ ] Restart Kernel and Run All works from top to bottom, and outputs are saved in the notebook
- [ ] Commit and push: `git add . && git commit -m "Add SQL analysis" && git push`